# Fase 4 — PyTorch Prático

## 🎯 Objetivo
Dominar os blocos fundamentais do PyTorch para construir, treinar e avaliar redes neurais de forma idiomática.

Ao final deste notebook você será capaz de:
- Construir modelos com **nn.Module** e **nn.Sequential**
- Entender o que é **autograd** e como `requires_grad` controla o treino
- Usar corretamente `.train()`, `.eval()` e `torch.no_grad()`
- Compreender e aplicar **Xavier initialization**
- Montar um pipeline completo de treino em PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch versão: {torch.__version__}")

## nn.Module: A Base de Toda Rede Neural

Em PyTorch, toda rede neural é uma classe que **herda de `nn.Module`**. Isso fornece:
- Registro automático dos parâmetros (`.parameters()`)
- Método `forward()` que define o cálculo
- Compatibilidade com otimizadores e funções de loss

### Estrutura básica:
```python
class MinhaRede(nn.Module):
    def __init__(self):
        super().__init__()
        # Defina as camadas aqui

    def forward(self, x):
        # Defina o forward pass aqui
        return x
```

**nn.Sequential** é um atalho para empilhar camadas em ordem, sem precisar definir o forward explicitamente.

In [ ]:
# === Comparando nn.Module manual vs nn.Sequential ===

# Versão 1: Usando nn.Sequential (mais conciso)
modelo_seq = nn.Sequential(
    nn.Linear(2, 4),   # entrada: 2 features → 4 neurônios ocultos
    nn.ReLU(),
    nn.Linear(4, 3)    # 4 neurônios ocultos → 3 classes
)

# Versão 2: Usando nn.Module (mais flexível)
class RedeManual(nn.Module):
    def __init__(self):
        super().__init__()
        self.camada_oculta  = nn.Linear(2, 4)
        self.ativacao       = nn.ReLU()
        self.camada_decisao = nn.Linear(4, 3)

    def forward(self, x):
        x = self.camada_oculta(x)
        x = self.ativacao(x)
        x = self.camada_decisao(x)
        return x

modelo_manual = RedeManual()

print("=== Arquitetura Sequential ===")
print(modelo_seq)
print(f"\nTotal de parâmetros: {sum(p.numel() for p in modelo_seq.parameters())}")
print()

print("=== Arquitetura Manual ===")
print(modelo_manual)

print("\n=== Parâmetros por camada ===")
for nome, param in modelo_manual.named_parameters():
    print(f"  {nome:30s}: shape={list(param.shape)}, n={param.numel()}")

# Verificar que produzem o mesmo tipo de saída
x_teste = torch.FloatTensor([[0.85, 0.85]])
print(f"\nSaída (Sequential): {modelo_seq(x_teste).detach().numpy().round(4)}")
print(f"Saída (Manual)     : {modelo_manual(x_teste).detach().numpy().round(4)}")

## requires_grad e Autograd

O **autograd** é o sistema de diferenciação automática do PyTorch. Para cada tensor com `requires_grad=True`, PyTorch constrói um **grafo computacional** e calcula gradientes automaticamente com `.backward()`.

### O que `requires_grad` controla:

| Cenário | `requires_grad` | Efeito |
|---|---|---|
| Parâmetro normal | `True` (padrão) | É atualizado pelo otimizador |
| Camada congelada | `False` | Não recebe gradiente, não é atualizado |
| Inferência (eval) | `False` (via `no_grad`) | Economiza memória, mais rápido |

> **Isso é a base do transfer learning:** você congela as camadas já treinadas e só treina as últimas.

In [ ]:
# === Demonstração do autograd ===

# Criando um tensor com gradiente
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Operação: y = w*x + b  (como um neurônio linear)
y = w * x + b
L = y ** 2  # "loss" simples para demonstrar

# Calculando gradientes
L.backward()

print("=== Autograd em ação ===")
print(f"x = {x.item()}, w = {w.item()}, b = {b.item()}")
print(f"y = w*x + b = {y.item()}")
print(f"L = y² = {L.item()}")
print()
print(f"dL/dw = 2*y*x = {w.grad.item():.4f}  (esperado: {(2*y*x).item():.4f})")
print(f"dL/db = 2*y   = {b.grad.item():.4f}  (esperado: {(2*y).item():.4f})")
print(f"dL/dx = 2*y*w = {x.grad.item():.4f}  (esperado: {(2*y*w).item():.4f})")

print("\n=== Congelando uma camada ===")
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 3)
)

# Congelar a primeira camada (como no notebook original!)
for param in model[0].parameters():
    param.requires_grad = False

print("Parâmetros por camada:")
for nome, param in model.named_parameters():
    estado = "🔒 CONGELADO" if not param.requires_grad else "🔓 treinável"
    print(f"  {nome:20s}: {estado}")

# Verificar que o otimizador só recebe os parâmetros treináveis
params_treinaveis = [p for p in model.parameters() if p.requires_grad]
print(f"\nParâmetros treináveis: {sum(p.numel() for p in params_treinaveis)}")
print(f"Parâmetros totais    : {sum(p.numel() for p in model.parameters())}")

## .train(), .eval() e torch.no_grad()

Esses três elementos controlam o **comportamento da rede em diferentes fases**:

| Modo / Context | Camadas afetadas | Uso |
|---|---|---|
| `model.train()` | Ativa Dropout, BatchNorm no modo treino | Durante o treino |
| `model.eval()` | Desativa Dropout, BatchNorm em modo inferência | Durante avaliação |
| `torch.no_grad()` | Desativa o cálculo de gradientes | Inferência/avaliação |

> **Importante:** `model.eval()` e `torch.no_grad()` são **independentes**.  
> `eval()` muda o comportamento de certas camadas (Dropout, BatchNorm).  
> `no_grad()` economiza memória e velocidade ao não construir o grafo computacional.

In [ ]:
# === Demonstração de train/eval e no_grad ===

import torch
import torch.nn as nn

# Modelo com Dropout para demonstrar a diferença
class ModeloComDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Dropout(p=0.5),  # desativa 50% dos neurônios aleatoriamente
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.net(x)

modelo = ModeloComDropout()
x = torch.ones(1, 4)

# MODO TREINO: Dropout ativo → saídas diferentes a cada chamada
modelo.train()
print("=== Modo TREINO (Dropout=0.5 ativo) ===")
for i in range(4):
    saida = modelo(x)
    print(f"  Chamada {i+1}: {saida.detach().numpy().round(4)}")
print("(Saídas DIFERENTES: Dropout desliga neurônios aleatoriamente)")

print()

# MODO AVALIAÇÃO: Dropout inativo → saída determinística
modelo.eval()
print("=== Modo AVALIAÇÃO (Dropout inativo) ===")
with torch.no_grad():
    for i in range(4):
        saida = modelo(x)
        print(f"  Chamada {i+1}: {saida.detach().numpy().round(4)}")
print("(Saídas IGUAIS: comportamento determinístico)")

print()

# Custo de memória: with vs without no_grad
x_grande = torch.randn(1000, 4, requires_grad=False)
modelo.eval()

# Sem no_grad: PyTorch guarda o grafo computacional
saida_com_grad = modelo(x_grande)
print(f"Com gradiente guardado    : tensor guarda grad_fn? {saida_com_grad.grad_fn is not None}")

# Com no_grad: mais eficiente
with torch.no_grad():
    saida_sem_grad = modelo(x_grande)
print(f"Com torch.no_grad()       : tensor guarda grad_fn? {saida_sem_grad.grad_fn is not None}")
print("→ no_grad() é mais eficiente para avaliação/inferência!")

## Xavier Initialization: Por Que e Como

Inicializar pesos de forma errada causa problemas sérios:
- **Pesos muito grandes** → ativações explodem → gradientes explodem
- **Pesos muito pequenos** → ativações somem → gradientes somem (vanishing)

### Fórmula de Xavier (Glorot):
$$w \sim U\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}, \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

**Intuição:** O desvio padrão é escolhido para que a **variância das ativações se mantenha constante** ao longo das camadas — nem explode, nem some.

> Para ReLU, usa-se **He initialization** (fator $\sqrt{2/n_{\text{in}}}$), que leva em conta que ReLU corta metade das ativações.

In [ ]:
# === Demonstração visual: efeito da inicialização ===
torch.manual_seed(42)
n_layers = 50   # rede bem profunda para amplificar o efeito

def simulacao_ativacoes(tipo_init, n_layers=50, n_neurons=100):
    ativacoes_std = []
    x = torch.randn(1000, n_neurons)

    for i in range(n_layers):
        if tipo_init == 'muito_pequena':
            W = torch.randn(n_neurons, n_neurons) * 0.01
        elif tipo_init == 'muito_grande':
            W = torch.randn(n_neurons, n_neurons) * 2.0
        elif tipo_init == 'xavier':
            std = np.sqrt(2.0 / (n_neurons + n_neurons))
            W = torch.randn(n_neurons, n_neurons) * std
        else:  # he
            std = np.sqrt(2.0 / n_neurons)
            W = torch.randn(n_neurons, n_neurons) * std

        x = torch.relu(x @ W)
        ativacoes_std.append(x.std().item())
        if x.std().item() > 1e6 or x.std().item() < 1e-6:
            break  # evitar overflow

    return ativacoes_std

fig, ax = plt.subplots(figsize=(10, 5))
configs = [
    ('muito_pequena', 'red',    'Inicialização pequena (std=0.01) → ativa. some'),
    ('muito_grande',  'orange', 'Inicialização grande (std=2.0)   → ativa. explode'),
    ('xavier',        'green',  'Xavier → ativações estáveis'),
    ('he',            'blue',   'He (para ReLU) → ativações estáveis'),
]
for tipo, cor, label in configs:
    std_list = simulacao_ativacoes(tipo)
    ax.plot(std_list[:n_layers], color=cor, linewidth=2, label=label)

ax.set_xlabel('Camada'); ax.set_ylabel('Desvio padrão das ativações')
ax.set_title('Efeito da inicialização em redes profundas', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Xavier na prática com PyTorch
print("=== Xavier no PyTorch ===")
linear = nn.Linear(2, 4)
print(f"Antes (default): std = {linear.weight.data.std():.4f}")

nn.init.xavier_uniform_(linear.weight)
linear.bias.data.zero_()
print(f"Depois (Xavier): std = {linear.weight.data.std():.4f}")
limite = np.sqrt(6.0 / (2 + 4))
print(f"Limite teórico: ±{limite:.4f}")

## Pipeline Completo em PyTorch

Juntando todos os conceitos em um exemplo completo e idiomático:

In [ ]:
# === Pipeline completo PyTorch — do zero ao modelo treinado ===
import torch, torch.nn as nn, torch.optim as optim

# --- Dados ---
np.random.seed(42)
N = 150
c1 = np.random.randn(N//3, 2) * 0.12 + [0.90, 0.90]
c2 = np.random.randn(N//3, 2) * 0.12 + [0.60, 0.60]
c3 = np.random.randn(N//3, 2) * 0.12 + [0.30, 0.30]
X_np = np.vstack([c1, c2, c3]).astype(np.float32)
Y_np = np.array([0]*(N//3) + [1]*(N//3) + [2]*(N//3))

X_tens = torch.from_numpy(X_np)
Y_tens = torch.from_numpy(Y_np).long()

# --- Modelo com Xavier init ---
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU(),
            nn.Linear(4, 3)
        )
        # Xavier initialization em todas as camadas Linear
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(x)

# === HIPERPARÂMETROS ===
model     = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
epochs    = 150

losses, accs = [], []

# --- Loop de treino ---
model.train()   # modo treino
for epoch in range(epochs):
    optimizer.zero_grad()           # 1. Zerar gradientes (obrigatório!)
    y_pred = model(X_tens)          # 2. Forward pass
    loss   = criterion(y_pred, Y_tens)  # 3. Calcular loss
    loss.backward()                 # 4. Backward pass (calcular gradientes)
    optimizer.step()                # 5. Atualizar pesos

    acc = (y_pred.argmax(dim=1) == Y_tens).float().mean().item()
    losses.append(loss.item()); accs.append(acc)

print(f"Treino final: Loss={losses[-1]:.4f}  Acurácia={accs[-1]:.4f}")

# --- Avaliação ---
model.eval()    # modo avaliação
with torch.no_grad():
    y_test = model(X_tens)
    loss_t = criterion(y_test, Y_tens)
    acc_t  = (y_test.argmax(dim=1) == Y_tens).float().mean().item()
print(f"Avaliação:   Loss={loss_t:.4f}  Acurácia={acc_t:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(losses, 'r-', lw=2); ax1.set_title('Loss'); ax1.grid(True, alpha=0.3)
ax1_b = ax1.twinx(); ax1_b.plot(accs, 'b--', lw=2, label='Acurácia')
ax1.set_xlabel('Época'); ax1.set_ylabel('Loss', color='red')
ax1_b.set_ylabel('Acurácia', color='blue')

xx, yy = np.meshgrid(np.linspace(0.1,1.2,200), np.linspace(0.1,1.2,200))
with torch.no_grad():
    Zp = model(torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])).argmax(1).numpy().reshape(xx.shape)
ax2.contourf(xx, yy, Zp, alpha=0.3, cmap='RdYlBu')
for cls, cor in enumerate(['red','green','blue']):
    mask = Y_np == cls
    ax2.scatter(X_np[mask,0], X_np[mask,1], c=cor, alpha=0.7, s=25, label=f'Classe {cls+1}')
ax2.set_title('Fronteiras — PyTorch'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()